# 09 — Demo en vivo: clasificar un chip con el modelo entrenado

Para la presentación: el profesor elige **cualquier PNG** del dataset (128×128).
Solo cambia `IMAGE_PATH` en la celda 2 y ejecuta todas las celdas.

**Requisitos previos:**
- Entorno activado: `pip install -e conf/`
- Checkpoint: `data/06_models/v2_bloques_tuned/resnet50_best.pt`
- Dataset en: `data/01_raw/dataset_amazonia_garimpo_binario/`

Alternativa rápida en terminal (sin notebook):
`garimpo predict-image ruta/al/chip.png`

## 1. Configuración

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

import matplotlib.pyplot as plt
from PIL import Image

from src.config import DEFAULT_CHECKPOINT, resolve_chips_dir
from src.models.predict import predict_image
from src.utils.helpers import get_device


CHIPS_DIR = resolve_chips_dir(PROJECT_ROOT / "data")
device = get_device()

print(f"Proyecto  : {PROJECT_ROOT}")
print(f"Chips     : {CHIPS_DIR}")
print(f"Device    : {device}")
print(f"Checkpoint: {DEFAULT_CHECKPOINT}")

if not DEFAULT_CHECKPOINT.exists():
    raise FileNotFoundError(
        f"Falta el checkpoint: {DEFAULT_CHECKPOINT}\n"
        "Copia resnet50_best.pt según data/06_models/v2_bloques_tuned/README.md"
    )

## 2. Elegir imagen (cambiar solo esta celda en la demo)

Pega la ruta relativa desde `dataset_amazonia_garimpo_binario/` **o** la ruta absoluta al PNG.

Ejemplos:
- `com_garimpo/archivo.png`
- `sem_garimpo/archivo.png`

In [ ]:
# <<< CAMBIAR AQUÍ EN LA PRESENTACIÓN >>>
IMAGE_PATH = CHIPS_DIR / "com_garimpo" / "v3_new_ret_s2_median_2024_dry-0000027136-0000027136_r0094_c0086_lat-8.02007_lon-54.90638_com_garimpo.png"

# Si el profesor da solo el nombre relativo (com_garimpo/...):
# IMAGE_PATH = CHIPS_DIR / "com_garimpo/nombre_del_chip.png"

IMAGE_PATH = Path(IMAGE_PATH)
if not IMAGE_PATH.is_absolute():
    IMAGE_PATH = CHIPS_DIR / IMAGE_PATH

if not IMAGE_PATH.exists():
    raise FileNotFoundError(f"No existe: {IMAGE_PATH}")

print(IMAGE_PATH)

## 3. Ver el chip

In [ ]:
with Image.open(IMAGE_PATH) as img:
    chip = img.convert("RGB")

plt.figure(figsize=(4, 4))
plt.imshow(chip)
plt.axis("off")
plt.title(IMAGE_PATH.name, fontsize=9)
plt.show()

## 4. Predicción

In [ ]:
result = predict_image(IMAGE_PATH, device=device)

label_es = {
    "com_garimpo": "CON minería (garimpo)",
    "sem_garimpo": "SIN minería (selva)",
}

print("=" * 50)
print(f"Modelo     : {result['model_name']}")
print(f"Predicción : {label_es[result['label_name']]}")
print(f"Clase      : {result['label_name']}")
print(f"P(garimpo) : {result['prob_com_garimpo']:.1%}")
print(f"P(selva)   : {result['prob_sem_garimpo']:.1%}")
print(f"Umbral     : {result['threshold']}")
print("=" * 50)

# Etiqueta real si viene en el nombre del archivo (solo referencia en demo)
name = IMAGE_PATH.name.lower()
if "com_garimpo" in name:
    print("Etiqueta en filename: com_garimpo (con minería)")
elif "sem_garimpo" in name:
    print("Etiqueta en filename: sem_garimpo (sin minería)")

result